# Report: what drives daily bike-share demand, and can we predict it?

This notebook only *tells the story*. All logic is imported from the `ridership` package, and every sentence containing numbers is **generated from the results**
so the text cannot drift from the analysis. (Data are synthetic; see the README.)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ridership import analysis, model, plots, report
from ridership.clean import clean_rides
from ridership.features import add_calendar_features, add_rain_flag
from ridership.load import load_rides

df = add_rain_flag(add_calendar_features(clean_rides(load_rides("../data/raw/daily_rides.csv"))))
print(len(df), "days;", int(df["temp_was_missing"].sum()), "missing temperatures filled with their month's median")

## Summary
Ridership follows the seasons and the weather. Temperature dominates, weekends are quieter, and rain only shows its effect once temperature is accounted for.
A simple linear model predicts unseen days well, and a random forest does no better. Details and uncertainty below.

## 1. Describe: level and season

In [ ]:
d = analysis.describe_rides(df)
print(report.describe_sentence(d))
plots.plot_monthly_rides(df); plt.show()

## 2. Compare: weekends vs weekdays

In [ ]:
c = analysis.compare_weekend(df)
print(report.weekend_sentence(c))
plots.plot_weekend_vs_weekday(df); plt.show()

## 3. Relate: temperature and rain

In [ ]:
corr = analysis.correlation_with_ci(df["temp_c"], df["rides"])
simple_slope = np.polyfit(df["temp_c"], df["rides"], 1)[0]
print(report.temperature_sentence(corr, simple_slope))
plots.plot_temperature_vs_rides(df, corr["r"]); plt.show()

In [ ]:
fit = analysis.fit_demand_model(df)
coefs = analysis.coefficient_table(fit)
display(coefs)
print(report.model_sentence(coefs, df["rain_mm"].corr(df["rides"])))
plots.plot_rain_effect(df, coefs.loc["rain_mm", "coef"]); plt.show()

**Check:** is this driven by the six filled-in temperatures? Refitting on observed temperatures only:

In [ ]:
observed = df[~df["temp_was_missing"]]
print(f"temperature slope with imputed days: {coefs.loc['temp_c', 'coef']:.1f}  |  without them: {analysis.fit_demand_model(observed).params['temp_c']:.1f}")

## 4. Predict: can we forecast unseen days?
We train on the first 306 days and test on the final 60 (chronological, so the model never sees the future), then repeat with expanding-window cross-validation.

In [ ]:
train, test = model.chronological_split(df, n_test=60)
eval_df = model.evaluate_models(train, test)
cv_df = model.time_series_cv_rmse(df)
display(eval_df.round(2)); display(cv_df.round(1))
print(report.prediction_sentence(eval_df, cv_df, len(test)))

In [ ]:
from sklearn.linear_model import LinearRegression

lin = LinearRegression().fit(train[model.FEATURES], train["rides"])
plots.plot_test_predictions(test, lin.predict(test[model.FEATURES]), eval_df.loc["linear regression", "rmse"]); plt.show()

## Limitations
* The data are **synthetic** and cover **one year** (one winter, one summer): we cannot separate a seasonal effect from a one-off trend, and real data would include holidays, events and growth.
* Results describe **associations**; we make no causal claims.
* Six temperatures were imputed; the sensitivity check suggests this does not matter here.
* The model is only trustworthy within the temperature range observed.

## Conclusion and next steps
Weather and weekends explain almost all of the variation in daily rides (R² ≈ 0.95), and a three-variable linear model forecasts the next two months to within about 100 rides.
Next: repeat with real trip data across several years, add holidays, and check whether errors are larger on unusual days.